# Silver — organisation hierarchy

Produces the three organisation columns that no SYSPER column holds directly: **Institution**, **DG** and **Entity**.

`sp2_org_unit_versions` stores each organisational unit with a link to its parent (`ou_id_admin_parent`). A unit does not
carry its DG or its Institution: they are found by walking up the parent links until the top.

Units are reorganised, so every row has a validity period (`date_from` / `date_to`). At each level of the walk, only the
parent versions valid **at the same time** as the child are kept. A unit can therefore belong to different DGs, under
different names, over time.

| | |
|---|---|
| **Reads** | `slv_org_unit_versions` |
| **Writes** | `slv_org_unit_hierarchy` |
| **Default lakehouse** | `lh_hrt_silver` |

**Output** — one row per `ou_id` per period during which the three values stay the same.

| Column | Value |
|---|---|
| `ou_id` | the organisational unit |
| `date_from`, `date_to` | the period; `date_to` is NULL while still valid, as in SYSPER |
| `ou_l0_acr` | `acr_compl` of the ancestor with `out_cd = 'INST'` |
| `ou_l1_acr` | `acr_compl` of the ancestor with `out_cd = 'DG'` |
| `ou_unit_acr` | `acr_compl` of the ancestor with `out_cd = 'UNIT'` |

A level that does not exist above a unit stays NULL. The business labels (Institution, DG, Entity) are given in Gold.

To get the picture on a given day: `date_from <= day AND (date_to IS NULL OR date_to >= day)`.

## Setup

In [ ]:
%run nb_config

In [ ]:
from datetime import date

from pyspark.sql import Window
from pyspark.sql.functions import (col, lit, when, coalesce, greatest, least,
                                   lag, datediff, date_add, sum as sum_, min as min_, max as max_)

spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")

# SYSPER leaves date_to empty for the version still valid. It is replaced by this
# date while comparing periods, and set back to NULL before writing.
OPEN_END = date(9999, 12, 31)

# The hierarchy is about six levels deep (INST > DG > ER3 > DIR > UNIT ...).
# A walk longer than this means the parent links contain a loop.
MAX_HIERARCHY_DEPTH = 15

---
## 1. Organisation versions

Only the five columns the walk needs. `date_from` / `date_to` must already be real dates: the Bronze values are Oracle
`DD-MON-YY` strings, and comparing periods on those strings would give wrong overlaps without any error.

In [ ]:
org = read_silver("slv_org_unit_versions")

date_types = {name: dtype for name, dtype in org.dtypes if name in ("date_from", "date_to")}
assert all(dtype in ("date", "timestamp") for dtype in date_types.values()), \
    f"date_from / date_to must be parsed as dates in nb_silver_clean, found {date_types}"

versions = (org
    .select("ou_id",
            "out_cd",
            "acr_compl",
            col("ou_id_admin_parent").alias("parent_id"),
            col("date_from").cast("date").alias("date_from"),
            coalesce(col("date_to").cast("date"), lit(OPEN_END)).alias("date_to"))
    .cache())

print(f"versions              {versions.count():>10,}")
print(f"organisational units  {versions.select('ou_id').distinct().count():>10,}")
versions.groupBy("out_cd").count().orderBy(col("count").desc()).show()

---
## 2. Periods without a parent

Every version except INST, which is the top, needs parent versions covering its whole period. Where days are missing, the
walk cannot go up and the unit is absent from the hierarchy on those days — the dashboard then shows an empty DG.

The check compares the number of days of each version with the number of days covered by its parent's versions.
Expected: 0.

In [ ]:
child = versions.filter(col("out_cd") != "INST").alias("child")
parent = versions.alias("parent")

coverage = (child
    .join(parent,
          (col("child.parent_id") == col("parent.ou_id"))
          & (col("parent.date_from") <= col("child.date_to"))
          & (col("parent.date_to") >= col("child.date_from")),
          "left")
    .groupBy("child.ou_id", "child.out_cd", "child.parent_id", "child.date_from", "child.date_to")
    .agg(sum_(datediff(least("child.date_to", "parent.date_to"),
                       greatest("child.date_from", "parent.date_from")) + 1).alias("days_covered")))

parent_gaps = (coverage
    .withColumn("days_in_period", datediff("date_to", "date_from") + 1)
    .filter(coalesce(col("days_covered"), lit(0)) != col("days_in_period"))
    .withColumn("date_to", when(col("date_to") != OPEN_END, col("date_to"))))

print(f"versions not fully covered by their parent  {parent_gaps.count():>6,}")
parent_gaps.orderBy("ou_id", "date_from").show(20, truncate=False)

---
## 3. Walk up the hierarchy

Each round replaces the current node by its parent:

- a period is matched only with the parent versions valid during it, and **shrinks to the overlap**. This is where a
  reorganisation higher up splits the period of a unit;
- when the node is an INST, a DG or a UNIT, its `acr_compl` goes into the matching column. The closest one wins;
- when the node is an INST, the path is complete and leaves the loop.

The first round matches every version with itself, so a DG row gets its own acronym as DG. Levels with no column of
their own, such as `DIR` or `ER3`, are passed through.

In [ ]:
to_visit = versions.select(
    "ou_id",
    "date_from",
    "date_to",
    col("ou_id").alias("next_id"),
    lit(None).cast("string").alias("ou_l0_acr"),
    lit(None).cast("string").alias("ou_l1_acr"),
    lit(None).cast("string").alias("ou_unit_acr"),
)

node = versions.alias("node")
complete = []

for level in range(1, MAX_HIERARCHY_DEPTH + 1):
    path = to_visit.alias("path")

    step = (path
        .join(node,
              (col("path.next_id") == col("node.ou_id"))
              & (col("node.date_from") <= col("path.date_to"))
              & (col("node.date_to") >= col("path.date_from")))
        .select(col("path.ou_id"),
                greatest("path.date_from", "node.date_from").alias("date_from"),
                least("path.date_to", "node.date_to").alias("date_to"),
                col("node.parent_id").alias("next_id"),
                col("node.out_cd"),
                coalesce("path.ou_l0_acr", when(col("node.out_cd") == "INST", col("node.acr_compl"))).alias("ou_l0_acr"),
                coalesce("path.ou_l1_acr", when(col("node.out_cd") == "DG", col("node.acr_compl"))).alias("ou_l1_acr"),
                coalesce("path.ou_unit_acr", when(col("node.out_cd") == "UNIT", col("node.acr_compl"))).alias("ou_unit_acr"))
        .localCheckpoint())  # each round builds on the previous one; this keeps the plan short

    complete.append(step.filter(col("out_cd") == "INST").drop("out_cd", "next_id"))
    to_visit = step.filter(col("out_cd") != "INST").drop("out_cd")

    remaining = to_visit.count()
    print(f"level {level:>2}   periods still walking up  {remaining:>10,}")
    if remaining == 0:
        break
else:
    raise RuntimeError(f"Still walking after {MAX_HIERARCHY_DEPTH} levels: check ou_id_admin_parent for a loop.")

paths = complete[0]
for part in complete[1:]:
    paths = paths.unionByName(part)

---
## 4. Merge the periods where nothing visible changed

A new version of an intermediate level (`DIR`, `ER3`...) splits the period during the walk, even when Institution, DG and
Unit stay the same. Consecutive periods with identical values are put back together, and the open end goes back to NULL.

In [ ]:
by_ou = Window.partitionBy("ou_id").orderBy("date_from")

same_as_previous = (
    col("ou_l0_acr").eqNullSafe(lag("ou_l0_acr").over(by_ou))
    & col("ou_l1_acr").eqNullSafe(lag("ou_l1_acr").over(by_ou))
    & col("ou_unit_acr").eqNullSafe(lag("ou_unit_acr").over(by_ou))
    & (date_add(lag("date_to").over(by_ou), 1) == col("date_from"))
)

hierarchy = (paths
    .withColumn("starts_new_period", when(same_as_previous, 0).otherwise(1))
    .withColumn("period_nbr", sum_("starts_new_period").over(by_ou))
    .groupBy("ou_id", "period_nbr", "ou_l0_acr", "ou_l1_acr", "ou_unit_acr")
    .agg(min_("date_from").alias("date_from"), max_("date_to").alias("date_to"))
    .select("ou_id",
            "date_from",
            when(col("date_to") != OPEN_END, col("date_to")).alias("date_to"),
            "ou_l0_acr",
            "ou_l1_acr",
            "ou_unit_acr"))

write_silver(hierarchy, "slv_org_unit_hierarchy")

---
## 5. Checks

**Overlapping periods.** On a given day an `ou_id` must have one row at most, otherwise every fact joined to it is
counted twice. It happens if a unit's parent has two versions valid on the same day. Expected: 0.

**Units missing from the output.** Units present in SYSPER but absent here: their chain never reaches an INST. It should
match the units listed in §2.

In [ ]:
result = read_silver("slv_org_unit_hierarchy")

by_ou = Window.partitionBy("ou_id").orderBy("date_from")
overlaps = (result
    .withColumn("previous_date_to", lag(coalesce(col("date_to"), lit(OPEN_END))).over(by_ou))
    .filter(col("previous_date_to") >= col("date_from")))

missing = versions.select("ou_id").distinct().join(result.select("ou_id").distinct(), "ou_id", "left_anti")

print(f"rows                    {result.count():>10,}")
print(f"organisational units    {result.select('ou_id').distinct().count():>10,}")
print(f"overlapping periods     {overlaps.count():>10,}")
print(f"units missing           {missing.count():>10,}")

**A known example.** Unit `230475` (AGRI.H.4 today) was traced by hand from SYSPER. Expected: COMMISSION / AGRI on every
row, and these units:

| date_from | date_to | ou_unit_acr |
|---|---|---|
| 2005-01-01 | 2007-03-18 | AGRI.J.5 |
| 2007-03-19 | 2008-06-30 | AGRI.DGA I-J-K.J.5 |
| 2008-07-01 | 2016-12-31 | AGRI.DDG4.J.5 |
| 2017-01-01 | 2022-01-15 | AGRI.DDG3.H.5 |
| 2022-01-16 | 2023-01-31 | AGRI.H.5 |
| 2023-02-01 | NULL | AGRI.H.4 |

Change `CHECK_OU_ID` and `CHECK_DATE` to look at another unit, or at what the dashboard shows on a given day.

In [ ]:
CHECK_OU_ID = 230475
CHECK_DATE = "2023-12-01"

display(result.filter(col("ou_id") == CHECK_OU_ID).orderBy("date_from"))

display(result.filter((col("ou_id") == CHECK_OU_ID)
                      & (col("date_from") <= CHECK_DATE)
                      & (col("date_to").isNull() | (col("date_to") >= CHECK_DATE))))